# Fault-distance preprocessing pipeline

Use **Restart Kernel and Run All** to run this notebook from beginning to end.
It loads one COMTRADE recording, filters the phase signals, detects and windows the
fault event, calculates sequence magnitudes, normalizes the signals, and exports
a 12-channel tensor. The supplied 2,000 Hz recordings produce a `(12, 400)` tensor.

Edit the configuration cell to choose `RECORD_NAME`, line parameters, and optional
known fault time/distance. The default record is `1A_val1`; an unknown target is
saved as `NaN`. Outputs go to `output/notebook/<RECORD_NAME>/`.

## 1. Install dependencies and import libraries
The first cell installs packages into the selected notebook kernel.


**DI Validator local edition.** The original pipeline is retained. Select a COMTRADE recording in the next cell, then run all cells. Data comes from the local application; calculations run in this browser. Line parameters are examples until verified. The original repository notebook is unchanged.


In [ ]:
from di_faults import fetch_record, recordings

RECORD_NAME = "1A_val1"  # Choose a name from await recordings().
DATA_FOLDER, source_metadata = await fetch_record(RECORD_NAME)


: 

The CFG/DAT pair is copied into browser storage. DAT bytes are unchanged. Legacy two-digit years are expanded using the documented 2000 + YY assumption; timezone remains unverified. No Google Drive account or external model service is used.


NumPy, pandas, SciPy and Matplotlib are hosted locally with Pyodide. The bundled COMTRADE reader is version 0.1.2 (MIT; see COMTRADE-LICENSE.md).


In [ ]:
import pyodide_js
for package in ["numpy", "pandas", "scipy", "matplotlib"]:
    await pyodide_js.loadPackage(package)

import json
import re
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
from comtrade import Comtrade
from scipy import signal

matplotlib.use("Agg")
import matplotlib.pyplot as plt

%matplotlib inline

plt.rcParams.update({
    "figure.figsize": (14, 6),
    "font.size": 10,
    "grid.alpha": 0.3,
    "savefig.dpi": 300,
})
print("Libraries loaded")


## 2. Configure the recording and output folder


In [ ]:
PROJECT_ROOT = Path.cwd()
OUTPUT_FOLDER = PROJECT_ROOT / "fault_outputs" / RECORD_NAME

# Example line/base values: set these to the actual line before interpreting p.u. values.
S_BASE_MVA = 100.0
LINE_PARAMS = {
    "Unom_kv": 110.0,
    "L_km": 50.0,
    "r1_ohm_km": 0.1,
    "x1_ohm_km": 0.4,
    "r0_ohm_km": None,  # Optional zero-sequence values for ground compensation.
    "x0_ohm_km": None,
}
FAULT_DISTANCE_KM = None  # Set only when the true distance is known.
FAULT_TIME_S = None      # Optional known inception time, measured from record start.
T_PRE = 0.050
T_POST = 0.150


def find_comtrade_files(folder_path: Path) -> list[dict[str, Path | str]]:
    """Find matching CFG/DAT pairs in a folder, sorted by basename."""
    pairs = []
    for cfg_file in sorted(folder_path.glob("*.cfg")):
        dat_file = cfg_file.with_suffix(".dat")
        if dat_file.is_file():
            pairs.append({"cfg": cfg_file, "dat": dat_file, "name": cfg_file.stem})
    return pairs


comtrade_files = find_comtrade_files(DATA_FOLDER)
if not comtrade_files:
    raise FileNotFoundError(f"No matching .cfg/.dat pairs found in {DATA_FOLDER}")
selected_pair = next((pair for pair in comtrade_files if pair["name"] == RECORD_NAME), None)
if selected_pair is None:
    available = ", ".join(pair["name"] for pair in comtrade_files)
    raise FileNotFoundError(f"Record {RECORD_NAME!r} not found. Available records: {available}")

cfg_path = selected_pair["cfg"]
dat_path = selected_pair["dat"]
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Found {len(comtrade_files)} COMTRADE file pairs")
print(f"Selected: {cfg_path}")
print(f"Outputs:  {OUTPUT_FOLDER}")


## 3. Load the recording and select phase channels
The default mapping matches the six-channel files in `data/data_test`.


In [ ]:
def inspect_channels(rec: Comtrade) -> dict[int, str]:
    """List analog positions, names, units, and phase tokens for manual selection."""
    print(f"{'No.':<5} | {'Channel name':<25} | {'Phase':<5} | {'Unit':<5}")
    print("-" * 55)
    found_info = {}
    for channel_number, channel in enumerate(rec.cfg.analog_channels, start=1):
        match = re.search(r"[IU]([ABC])(?:PRIM|SEC)?$", channel.name.upper().strip())
        phase = match.group(1) if match else "?"
        print(f"{channel_number:<5} | {channel.name:<25} | {phase:<5} | {channel.uu:<5}")
        found_info[channel_number] = channel.name
    return found_info


rec = Comtrade(use_double_precision=True)
rec.load(str(cfg_path), str(dat_path))
# Preserve supplied DAT timing; the reader otherwise reconstructs it from Fs.
native_times = np.loadtxt(dat_path, delimiter=",", usecols=1) * rec.cfg.timemult * rec.cfg.time_base
for index, stamp in enumerate(native_times):
    rec.time[index] = float(stamp)
if not np.allclose(np.diff(native_times), 1 / rec.cfg.sample_rates[0][0], rtol=0, atol=1e-9):
    raise ValueError("This notebook requires a contiguous recording. Use Event Lab to analyze segments separated by gaps.")
channel_map = inspect_channels(rec)
if len(rec.cfg.sample_rates) != 1:
    raise ValueError("This notebook requires a recording with one constant sampling rate.")
FS = float(rec.cfg.sample_rates[0][0])
F_NET = float(rec.cfg.frequency)
if not 0 < F_NET < FS / 2:
    raise ValueError(f"Invalid sampling/grid frequencies: fs={FS}, f_net={F_NET}")
if len(rec.time) < 2 * int(round(FS / F_NET)):
    raise ValueError("The recording must contain at least two grid periods.")
print(f"Samples: {len(rec.time)}, sampling rate: {FS:g} Hz, grid frequency: {F_NET:g} Hz")


In [ ]:
def parse_comtrade(
    rec: Comtrade,
    i_prim_nums: list[int],
    i_sec_nums: list[int],
    u_prim_nums: list[int],
    u_sec_nums: list[int],
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Build phase matrices from 1-based analog channel numbers in A, B, C order.

    Each group must contain three channel numbers or be empty. An empty group
    returns a (3, N) zero placeholder; it does not represent measured signals.
    """
    channel_count = len(rec.analog)

    def get_data_by_nums(nums: list[int], group: str) -> np.ndarray:
        if len(nums) not in (0, 3):
            raise ValueError(f"{group}: select exactly 3 channels (A, B, C), or use [].")
        invalid = [
            num for num in nums
            if isinstance(num, (bool, np.bool_))
            or not isinstance(num, (int, np.integer))
            or not 1 <= num <= channel_count
        ]
        if invalid:
            available = ", ".join(
                f"{index}: {name}"
                for index, name in enumerate(rec.analog_channel_ids, start=1)
            )
            raise ValueError(
                f"{group}: invalid channel numbers {invalid}. This recording has "
                f"{channel_count} analog channels; select integers from 1 to {channel_count}. "
                f"Available channels: {available}. Use [] for an absent group."
            )
        mat = np.zeros((3, len(rec.time)))
        for phase_index, num in enumerate(nums):
            mat[phase_index] = rec.analog[num - 1]
        return mat

    i_prim = get_data_by_nums(i_prim_nums, "i_prim_nums")
    i_sec = get_data_by_nums(i_sec_nums, "i_sec_nums")
    u_prim = get_data_by_nums(u_prim_nums, "u_prim_nums")
    u_sec = get_data_by_nums(u_sec_nums, "u_sec_nums")

    print("✓ Arrays I_prim, I_sec, U_prim, U_sec created successfully.")
    return i_prim, i_sec, u_prim, u_sec

# Mapping for the six-channel recordings in data/data_test, in A, B, C order.
# Check inspect_channels(rec) and update these numbers for a different channel layout.
i_p_idx = [1, 2, 3]  # CT1IA, CT1IB, CT1IC (kA)
i_s_idx = []         # No secondary currents recorded; zero placeholders only.
u_p_idx = [4, 5, 6]  # S1) BUS1UA, S1) BUS1UB, S1) BUS1UC (kV)
u_s_idx = []         # No secondary voltages recorded; zero placeholders only.

if len(i_p_idx) != 3 or len(u_p_idx) != 3:
    raise ValueError("Select all three primary current and voltage channels for this pipeline.")

I_prim, I_sec, U_prim, U_sec = parse_comtrade(
    rec,
    i_prim_nums=i_p_idx,
    i_sec_nums=i_s_idx,
    u_prim_nums=u_p_idx,
    u_sec_nums=u_s_idx,
)

# === Build a shared data context for subsequent sections ===
IA_filtered = I_prim[0]
IB_filtered = I_prim[1]
IC_filtered = I_prim[2]
UA_filtered = U_prim[0]
UB_filtered = U_prim[1]
UC_filtered = U_prim[2]

# Line parameters and nominal values
Inom_kA = S_BASE_MVA / (np.sqrt(3) * LINE_PARAMS["Unom_kv"])

data = {
    'fs': FS,
    'frequency': F_NET,
    'Inom': Inom_kA,
    'fault_time': FAULT_TIME_S,
    'fault_distance_km': FAULT_DISTANCE_KM,
}
print("✓ Data context created")


## 4. Inspect, center, and filter the phase waveforms
Analysis uses kA and kV; source A/V channels are converted before filtering.


In [ ]:
def plot_phases(rec, i_data, u_data, i_nums, u_nums, title="Waveforms"):
    """
    Plot phases A, B, C with channel names and units from the CFG file.
    """
    fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex=True)

    time_ms = np.array(rec.time) * 1000
    colors = ['#d62728', '#2ca02c', '#1f77b4'] # A, B, C

    # Helper function to retrieve metadata (name and unit)
    def get_meta(nums, idx_in_list):
        try:
            ch_idx = nums[idx_in_list] - 1
            channel = rec.cfg.analog_channels[ch_idx]
            return channel.name, channel.uu
        except:
            return f"Channel {idx_in_list}", "?"

    for row in range(3):
        # --- CURRENTS (left column) ---
        name_i, unit_i = get_meta(i_nums, row)
        ax_i = axes[row, 0]
        ax_i.plot(time_ms, i_data[row], color=colors[row], lw=1.5)
        ax_i.set_ylabel(f"[{unit_i}]")
        ax_i.set_title(f"{name_i}", loc='left', fontweight='bold', fontsize=11)
        ax_i.grid(True, alpha=0.3)

        # --- VOLTAGES (right column) ---
        name_u, unit_u = get_meta(u_nums, row)
        ax_u = axes[row, 1]
        ax_u.plot(time_ms, u_data[row], color=colors[row], lw=1.5)
        ax_u.set_ylabel(f"[{unit_u}]")
        ax_u.set_title(f"{name_u}", loc='left', fontweight='bold', fontsize=11)
        ax_u.grid(True, alpha=0.3)

    # Shared X-axis labels
    axes[2, 0].set_xlabel("Time [ms]")
    axes[2, 1].set_xlabel("Time [ms]")

    fig.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# Usage (ensure i_p_idx and u_p_idx are defined in the cell above)
plot_phases(rec, I_prim, U_prim, i_p_idx, u_p_idx, title="Primary quantities")


In [ ]:
ORDER = 4
FC = 25.0


def to_kilo_units(values: np.ndarray, nums: list[int], quantity: str) -> np.ndarray:
    """Convert recorded A/kA or V/kV phase channels to kA or kV."""
    scales = {"A": 1e-3, "kA": 1.0} if quantity == "I" else {"V": 1e-3, "kV": 1.0}
    factors = []
    for number in nums:
        unit = rec.cfg.analog_channels[number - 1].uu.strip()
        if unit not in scales:
            raise ValueError(f"Unsupported unit {unit!r} for {quantity} channel {number}")
        factors.append(scales[unit])
    return values * np.asarray(factors)[:, None]


def remove_dc_period(values: np.ndarray, fs: float, f_net: float) -> np.ndarray:
    """Subtract a centered moving average over one grid period."""
    period = int(round(fs / f_net))
    filtered = np.zeros_like(values)
    for index, waveform in enumerate(values):
        aper = pd.Series(waveform).rolling(window=period, center=True, min_periods=1).mean()
        half = period // 2
        aper.iloc[:half] = aper.iloc[half:period].mean()
        aper.iloc[-half:] = aper.iloc[-period:-half].mean()
        filtered[index] = waveform - aper.to_numpy()
    return filtered


I_prim_kA = to_kilo_units(I_prim, i_p_idx, "I")
U_prim_kV = to_kilo_units(U_prim, u_p_idx, "U")
pre_window = int(round(FS / F_NET))
I_centered = I_prim_kA - I_prim_kA[:, :pre_window].mean(axis=1, keepdims=True)
U_centered = U_prim_kV - U_prim_kV[:, :pre_window].mean(axis=1, keepdims=True)
I_prim_filt = remove_dc_period(I_centered, FS, F_NET)
U_prim_filt = remove_dc_period(U_centered, FS, F_NET)

IA_filtered, IB_filtered, IC_filtered = I_prim_filt
UA_filtered, UB_filtered, UC_filtered = U_prim_filt
print("Filtering complete: prehistory centering and one-period moving-average subtraction")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
t_ms = np.array(rec.time) * 1000

# 1. Phase A current: before and after filtering
axes[0, 0].plot(t_ms, I_prim_kA[0], 'gray', alpha=0.5, label='Original')
axes[0, 0].plot(t_ms, I_prim_filt[0], 'r-', label='Filtered')
axes[0, 0].set_ylabel('IA, kA')
axes[0, 0].set_title('Phase A current: before and after filtering')
axes[0, 0].legend()
axes[0, 0].grid(True)

# 2. DC component (moving average over one grid period)
window_size = int(round(FS / F_NET))
dc_before = np.convolve(I_prim_kA[0], np.ones(window_size)/window_size, mode='same')
dc_after = np.convolve(I_prim_filt[0], np.ones(window_size)/window_size, mode='same')

axes[1, 0].plot(t_ms, dc_before, 'gray', alpha=0.7, label='DC before')
axes[1, 0].plot(t_ms, dc_after, 'b-', label='DC after')
axes[1, 0].set_ylabel('Mean, kA')
axes[1, 0].set_title('DC component (IA)')
axes[1, 0].legend()
axes[1, 0].grid(True)

# 3. Butterworth filter magnitude response (for reference)
sos_ref = signal.butter(ORDER, FC/(0.5*FS), btype='highpass', output='sos')
w, h = signal.sosfreqz(sos_ref, worN=2000, fs=FS)
axes[0, 1].semilogx(w, 20*np.log10(np.abs(h) + 1e-10), 'b-', linewidth=2)
axes[0, 1].axvline(x=FC, color='red', linestyle='--', label=f'fc = {FC} Hz')
axes[0, 1].axvline(x=F_NET, color='green', linestyle='--', label=f'{F_NET:g} Hz')
axes[0, 1].set_ylabel('Magnitude, dB')
axes[0, 1].set_title('Reference Butterworth HPF magnitude response')
axes[0, 1].legend()
axes[0, 1].grid(True, which='both')
axes[0, 1].set_ylim([-60, 5])

# 4. ALL CURRENTS after filtering
axes[1, 1].plot(t_ms, I_prim_filt[0], 'r-', label='IA', lw=1)
axes[1, 1].plot(t_ms, I_prim_filt[1], 'g-', label='IB', lw=1)
axes[1, 1].plot(t_ms, I_prim_filt[2], 'b-', label='IC', lw=1)
axes[1, 1].set_ylabel('Current, kA')
axes[1, 1].set_xlabel('Time, ms')
axes[1, 1].set_title('All phases after filtering')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()


## 5. Export filtered COMTRADE signals
The CFG/DAT pair uses integer ASCII samples and per-channel scaling.


In [ ]:
import numpy as np
file_name = OUTPUT_FOLDER / "filtered_data"

# 1. Build the CFG header in strict accordance with IEEE C37.111-1999
station_name = "FILTERED"
rec_dev_id = "PY_SCRIPT"
rev_year = "1999"
nA = 6
nD = 0
tt = nA + nD

cfg_lines = [
    f"{station_name},{rec_dev_id},{rev_year}",
    f"{tt},{nA}A,{nD}D",
]

all_signals = [
    np.array(I_prim_filt[0]), np.array(I_prim_filt[1]), np.array(I_prim_filt[2]),
    np.array(U_prim_filt[0]), np.array(U_prim_filt[1]), np.array(U_prim_filt[2])
]
names = ["IA_filt", "IB_filt", "IC_filt", "UA_filt", "UB_filt", "UC_filt"]
phases = ["A", "B", "C", "A", "B", "C"]
units = ["kA", "kA", "kA", "kV", "kV", "kV"]

for i in range(6):
    sig = all_signals[i]
    ch_id = names[i]
    ph = phases[i]
    ccbm = ""
    uu = units[i]
    a = 1.0
    b = 0.0
    skew = 0
    min_val = -32767
    max_val = 32767
    a = max(float(np.max(np.abs(sig))) / max_val, 1e-12)
    primary = 1
    secondary = 1
    ps = "P"
    cfg_lines.append(f"{i+1},{ch_id},{ph},{ccbm},{uu},{a},{b},{skew},{min_val},{max_val},{primary},{secondary},{ps}")

lf = rec.cfg.frequency
nrates = 1
npts = len(all_signals[0])
start_dt = rec.start_timestamp.strftime("%d/%m/%Y,%H:%M:%S.%f")
trigger_dt = rec.trigger_timestamp.strftime("%d/%m/%Y,%H:%M:%S.%f")

cfg_lines.extend([
    str(lf),
    str(nrates),
    f"{FS:g},{npts}",
    start_dt,
    trigger_dt,
    "ASCII",
    "1"
])

with open(f"{file_name}.cfg", "w", newline="") as f:
    f.write("\n".join(cfg_lines) + "\n")

# 2. Build DAT (ASCII)
scales = [max(float(np.max(np.abs(sig))) / 32767, 1e-12) for sig in all_signals]
with open(f"{file_name}.dat", "w", newline="") as f:
    for i in range(npts):
        t_us = int(round((rec.time[i] - rec.time[0]) * 1_000_000))
        vals = ",".join(str(int(np.rint(sig[i] / scale)))
                        for sig, scale in zip(all_signals, scales))
        f.write(f"{i+1},{t_us},{vals}\n")

print(f"✅ Files created: {file_name}.cfg and {file_name}.dat (IEEE C37.111-1999)")


## 6. Detect fault inception and inspect residual DC


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

def analyze_signals_full(data_dict, fs, f_net, fault_idx=None, threshold=5.0):
    """
    Analyze the aperiodic component.
    Estimate DC% using the pre-fault history (before fault inception),
    where filtering should completely suppress DC.
    """
    window_size = int(round(fs / f_net))
    half_w = window_size // 2
    results = {}

    names = list(data_dict.keys())
    fig, axes = plt.subplots(len(names), 1, figsize=(12, 4 * len(names)))
    if len(names) == 1:
        axes = [axes]

    print(f"{'Channel':<12} | {'DC %':<10} | {'Ta (ms)':<10} | {'Status'}")
    print("-" * 50)

    for i, name in enumerate(names):
        signal_full = np.array(data_dict[name]).flatten()

        # Use the pre-fault history or the entire signal to estimate DC
        if fault_idx is not None and fault_idx > window_size:
            eval_sig = signal_full[:fault_idx]
        else:
            eval_sig = signal_full

        if len(eval_sig) > window_size:
            aper_eval = uniform_filter1d(eval_sig, size=window_size, mode='nearest')
            sig_eval = eval_sig[half_w:-half_w]
            aper_eval = aper_eval[half_w:-half_w]
        else:
            aper_eval = uniform_filter1d(eval_sig, size=len(eval_sig), mode='nearest')
            sig_eval = eval_sig

        ac_eval = sig_eval - aper_eval
        peak_ac = np.max(np.abs(ac_eval))
        max_dc = np.max(np.abs(aper_eval))
        ratio = (max_dc / peak_ac * 100) if peak_ac > 0 else 0

        abs_dc = np.abs(aper_eval)
        idx_peak = np.argmax(abs_dc)
        val_peak = abs_dc[idx_peak]
        target = val_peak * 0.368
        ta = 0
        fragment = abs_dc[idx_peak:]
        for j, val in enumerate(fragment):
            if val <= target:
                ta = (j / fs) * 1000
                break

        results[name] = {"ratio": ratio, "ta": ta}
        status = "⚠️ HIGH" if ratio > threshold else "✅ NORMAL"
        ta_display = f"{ta:.1f}" if ta > 0 else "> window"
        print(f"{name:<12} | {ratio:>7.2f}% | {ta_display:>9} | {status}")

        ax = axes[i]
        t_full = np.arange(len(signal_full)) / fs * 1000
        aper_full = uniform_filter1d(signal_full, size=window_size, mode='nearest')
        ax.plot(t_full, signal_full, color='gray', alpha=0.4, label='Original signal')
        ax.plot(t_full, aper_full, color='red', linewidth=2, label='DC (aperiodic)')
        ax.plot(t_full, signal_full - aper_full, color='blue', alpha=0.7, label='AC (fundamental)')
        if fault_idx is not None:
            ax.axvline(x=fault_idx/fs*1000, color='magenta', linestyle='--', alpha=0.5, label='Fault')
        ax.axvspan(0, half_w/fs*1000, color='yellow', alpha=0.1)
        ax.axvspan(t_full[-1] - half_w/fs*1000, t_full[-1], color='yellow', alpha=0.1)
        ax.set_title(f"Channel analysis: {name} (DC={ratio:.1f}%, Ta={ta_display} ms)")
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return results

data_to_test = {
    "IA_filt": np.array(I_prim_filt[0]).flatten(),
    "IB_filt": np.array(I_prim_filt[1]).flatten(),
    "IC_filt": np.array(I_prim_filt[2]).flatten(),
    "UA_filt": np.array(U_prim_filt[0]).flatten(),
    "UB_filt": np.array(U_prim_filt[1]).flatten(),
    "UC_filt": np.array(U_prim_filt[2]).flatten()
}


In [ ]:
def detect_fault_inception(ia, ib, ic, ua, ub, uc, fs, f_net, Inom, eta_I=0.5, eta_U=0.85):
    """Detect fault inception using a combined sliding-window RMS criterion.

    The algorithm compares current and voltage RMS in two adjacent windows
    of one-quarter period each (k = fs/(4*f_net)).
    A fault is detected when the current rises sharply in at least one phase
    (I_post / I_pre > 1 + eta_I) and the voltage drops
    (U_post / U_pre < eta_U).
    """
    k = int(fs / (4 * f_net))
    n = len(ia)

    def window_rms(x, start, end):
        return np.sqrt(np.mean(x[start:end]**2))

    I_ratio = np.ones(n)
    U_ratio = np.ones(n)

    for i in range(k, n - k):
        I_pre = max(window_rms(ia, i - k, i),
                    window_rms(ib, i - k, i),
                    window_rms(ic, i - k, i))
        I_post = max(window_rms(ia, i, i + k),
                     window_rms(ib, i, i + k),
                     window_rms(ic, i, i + k))
        U_pre = min(window_rms(ua, i - k, i),
                    window_rms(ub, i - k, i),
                    window_rms(uc, i - k, i))
        U_post = min(window_rms(ua, i, i + k),
                     window_rms(ub, i, i + k),
                     window_rms(uc, i, i + k))

        I_ratio[i] = I_post / (I_pre + 1e-9) if I_pre > 1e-6 else 1.0
        U_ratio[i] = U_post / (U_pre + 1e-9) if U_pre > 1e-6 else 1.0

    # Skip the first 2 windows to prevent false triggering
    # at the start of the recording (energization, switching surge)
    skip = 2 * k
    candidates = np.where((I_ratio > (1.0 + eta_I)) & (U_ratio < eta_U))[0]
    candidates = candidates[candidates >= skip]

    if len(candidates) == 0:
        return None, I_ratio, U_ratio

    fault_idx = candidates[0]
    return fault_idx, I_ratio, U_ratio

ETA_I = 0.5   # 50% current increase
ETA_U = 0.85  # voltage drops below 85%

fault_idx_detected, I_ratio, U_ratio = detect_fault_inception(
    IA_filtered, IB_filtered, IC_filtered,
    UA_filtered, UB_filtered, UC_filtered,
    data['fs'], data['frequency'], data['Inom'], ETA_I, ETA_U
)

fault_time_detected = fault_idx_detected / data['fs'] if fault_idx_detected is not None else None

print("✓ Fault detection complete")
print(f"  Current threshold: I_post / I_pre > {1.0 + ETA_I:.2f}")
print(f"  Voltage threshold: U_post / U_pre < {ETA_U:.2f}")
if fault_time_detected is not None:
    print(f"  Detected: sample {fault_idx_detected}, time {fault_time_detected*1000:.2f} ms")
    if data['fault_time'] is not None:
        print(f"  True time: {data['fault_time']*1000:.2f} ms")
        print(f"  Error: {abs(fault_time_detected - data['fault_time'])*1000:.2f} ms")
else:
    print("  ⚠️ No fault detected")

# Assess filtering quality using the pre-fault history
results = analyze_signals_full(
    data_to_test, fs=data['fs'], f_net=data['frequency'], fault_idx=fault_idx_detected
)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
t_ms = np.arange(len(IA_filtered)) / data['fs'] * 1000

axes[0].plot(t_ms, IA_filtered, 'r-', label='IA')
axes[0].plot(t_ms, IB_filtered, 'g-', label='IB')
axes[0].plot(t_ms, IC_filtered, 'b-', label='IC')
if data['fault_time'] is not None:
    axes[0].axvline(x=data['fault_time']*1000, color='cyan', linestyle='-', linewidth=2, label='True fault inception')
if fault_time_detected is not None:
    axes[0].axvline(x=fault_time_detected*1000, color='magenta', linestyle='--', linewidth=2, label='Detected')
axes[0].set_ylabel('Current, kA')
axes[0].set_title('Fault inception detection')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(t_ms, I_ratio, 'k-', label='$I_{post} / I_{pre}$')
axes[1].plot(t_ms, U_ratio, 'b--', label='$U_{post} / U_{pre}$')
axes[1].axhline(y=1.0 + ETA_I, color='red', linestyle='--', label='Current threshold')
axes[1].axhline(y=ETA_U, color='green', linestyle='--', label='Voltage threshold')
if data['fault_time'] is not None:
    axes[1].axvline(x=data['fault_time']*1000, color='cyan', linestyle='-', linewidth=2)
if fault_time_detected is not None:
    axes[1].axvline(x=fault_time_detected*1000, color='magenta', linestyle='--', linewidth=2)
    axes[1].scatter([fault_time_detected*1000], [I_ratio[fault_idx_detected]], color='magenta', s=100)
axes[1].set_ylabel('RMS ratio')
axes[1].set_xlabel('Time, ms')
axes[1].set_title('Current and voltage changes (sliding window = T/4)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

if fault_idx_detected is not None:
    print(f"Maximum current ratio: {np.max(I_ratio):.3f}")
    print(f"Minimum voltage ratio: {np.min(U_ratio):.3f}")


## 7. Extract the analysis window
A manual fault time takes priority; otherwise use the detected time, then the record midpoint.


In [ ]:
if T_PRE <= 0 or T_POST <= 0:
    raise ValueError('T_PRE and T_POST must be positive.')

N_PRE = int(round(T_PRE * data['fs']))
N_POST = int(round(T_POST * data['fs']))
if min(N_PRE, N_POST) < 1:
    raise ValueError('Each analysis window must contain at least one sample.')
L_WINDOW = N_PRE + N_POST

if FAULT_TIME_S is not None:
    window_center = int(round(FAULT_TIME_S * FS))
    if not 0 <= window_center < len(IA_filtered):
        raise ValueError("FAULT_TIME_S must fall inside the recording.")
    window_source = "manual inception time"
elif fault_idx_detected is not None:
    window_center = int(fault_idx_detected)
    window_source = "detected inception time"
else:
    window_center = len(IA_filtered) // 2
    window_source = "record midpoint (no fault detected)"
    print("No fault detected; centering the analysis window on the record midpoint.")

def extract_window(signal, center, n_pre, n_post):
    """Extract a fixed-length n_pre + n_post window, padding with zeros outside the bounds."""
    start = center - n_pre
    end = center + n_post
    pad_left = max(0, -start)
    pad_right = max(0, end - len(signal))
    valid_start = max(0, start)
    valid_end = min(len(signal), end)
    window = np.concatenate([
        np.zeros(pad_left),
        signal[valid_start:valid_end],
        np.zeros(pad_right)
    ])
    return window

IA_window = extract_window(IA_filtered, window_center, N_PRE, N_POST)
IB_window = extract_window(IB_filtered, window_center, N_PRE, N_POST)
IC_window = extract_window(IC_filtered, window_center, N_PRE, N_POST)
UA_window = extract_window(UA_filtered, window_center, N_PRE, N_POST)
UB_window = extract_window(UB_filtered, window_center, N_PRE, N_POST)
UC_window = extract_window(UC_filtered, window_center, N_PRE, N_POST)

t_window = np.arange(-N_PRE, N_POST) / data['fs'] * 1000

print(f"Windowing complete: {window_source}, sample {window_center}")
print(f"  Pre-fault history: {T_PRE*1000:.0f} ms = {N_PRE} samples")
print(f"  Post-fault history: {T_POST*1000:.0f} ms = {N_POST} samples")
print(f"  Total length: {L_WINDOW} samples")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(t_window, IA_window, 'r-', label='IA')
axes[0].plot(t_window, IB_window, 'g-', label='IB')
axes[0].plot(t_window, IC_window, 'b-', label='IC')
axes[0].axvline(x=0, color='magenta', linestyle='--', linewidth=2, label='Window center')
axes[0].axvspan(-T_PRE*1000, 0, alpha=0.1, color='green', label='Before window center')
axes[0].axvspan(0, T_POST*1000, alpha=0.1, color='red', label='After window center')
axes[0].set_ylabel('Current, kA')
axes[0].set_title('Currents in the analysis window')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(t_window, UA_window, 'r-', label='UA')
axes[1].plot(t_window, UB_window, 'g-', label='UB')
axes[1].plot(t_window, UC_window, 'b-', label='UC')
axes[1].axvline(x=0, color='magenta', linestyle='--', linewidth=2)
axes[1].axvspan(-T_PRE*1000, 0, alpha=0.1, color='green')
axes[1].axvspan(0, T_POST*1000, alpha=0.1, color='red')
axes[1].set_ylabel('Voltage, kV')
axes[1].set_xlabel('Time from window center, ms')
axes[1].set_title('Voltages in the analysis window')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"IA amplitude before the window center: {np.max(np.abs(IA_window[:N_PRE])):.2f} kA")
print(f"IA amplitude after the window center: {np.max(np.abs(IA_window[N_PRE:])):.2f} kA")


## 8. Calculate symmetrical-component magnitudes
A centered one-cycle DFT feeds the Fortescue transform. Values are peak magnitudes.


In [ ]:
def sliding_fortescue_magnitudes(abc: np.ndarray, fs: float, f0: float) -> np.ndarray:
    """Return one-cycle fundamental peak magnitudes in zero/positive/negative order.

    Args:
        abc: Finite signals with shape (3, N), in A/B/C order.
        fs: Sampling frequency in Hz.
        f0: Fundamental frequency in Hz.

    Returns:
        Array with shape (3, N): |X0|, |X1|, |X2|. Divide by sqrt(2) for RMS.

    Each centered DFT window spans one electrical period. Linear interpolation
    handles noninteger fs/f0. Outside the record, samples repeat the nearest
    endpoint; results within half a period of either edge are approximate.
    """
    abc = np.asarray(abc, dtype=float)
    if abc.ndim != 2 or abc.shape[0] != 3 or abc.shape[1] == 0:
        raise ValueError("abc must have shape (3, N) with N > 0.")
    if not np.isfinite(abc).all():
        raise ValueError("abc must contain only finite samples.")
    if not np.isfinite(fs) or not np.isfinite(f0) or f0 <= 0 or fs <= 2 * f0:
        raise ValueError("Require finite fs > 2*f0 > 0.")

    n_samples = abc.shape[1]
    n_cycle = max(3, int(round(fs / f0)))
    phase_steps = np.arange(n_cycle) - n_cycle // 2
    offsets = phase_steps * fs / (f0 * n_cycle)
    positions = np.arange(n_samples)[:, None] + offsets
    reference = np.exp(-2j * np.pi * phase_steps / n_cycle)
    phasors = np.stack([
        (2.0 / n_cycle)
        * (np.interp(positions, np.arange(n_samples), phase) @ reference)
        for phase in abc
    ])
    a = np.exp(2j * np.pi / 3)
    fortescue = np.array([
        [1, 1, 1],
        [1, a, a**2],
        [1, a**2, a],
    ]) / 3
    return np.abs(fortescue @ phasors)


# Estimate on the complete filtered record before cropping/padding the analysis window.
I_sequences = sliding_fortescue_magnitudes(I_prim_filt, FS, F_NET)
U_sequences = sliding_fortescue_magnitudes(U_prim_filt, FS, F_NET)
I0, I1, I2 = [extract_window(row, window_center, N_PRE, N_POST) for row in I_sequences]
U0, U1, U2 = [extract_window(row, window_center, N_PRE, N_POST) for row in U_sequences]
print("Symmetrical components calculated: one-cycle DFT peak magnitudes")


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

axes[0, 0].plot(t_window, I0, 'purple', label='|I₀|')
axes[0, 0].plot(t_window, I1, 'blue', label='|I₁|')
axes[0, 0].plot(t_window, I2, 'orange', label='|I₂|')
axes[0, 0].axvline(x=0, color='magenta', linestyle='--')
axes[0, 0].set_ylabel('Current, kA')
axes[0, 0].set_title('Symmetrical current magnitudes (peak)')
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(t_window, U0, 'purple', label='|U₀|')
axes[0, 1].plot(t_window, U1, 'blue', label='|U₁|')
axes[0, 1].plot(t_window, U2, 'orange', label='|U₂|')
axes[0, 1].axvline(x=0, color='magenta', linestyle='--')
axes[0, 1].set_ylabel('Voltage, kV')
axes[0, 1].set_title('Symmetrical voltage magnitudes (peak)')
axes[0, 1].legend()
axes[0, 1].grid(True)

axes[1, 0].plot(t_window, IA_window, 'r-', alpha=0.5, label='IA')
axes[1, 0].plot(t_window, IB_window, 'g-', alpha=0.5, label='IB')
axes[1, 0].plot(t_window, IC_window, 'b-', alpha=0.5, label='IC')
axes[1, 0].plot(t_window, I1, 'k-', linewidth=2, label='|I₁|')
axes[1, 0].axvline(x=0, color='magenta', linestyle='--')
axes[1, 0].set_ylabel('Current, kA')
axes[1, 0].set_title('Phase currents and |I₁|')
axes[1, 0].legend()
axes[1, 0].grid(True)

sum_abc = (IA_window + IB_window + IC_window) / 3
axes[1, 1].plot(t_window, sum_abc, 'gray', alpha=0.7, label='(IA+IB+IC)/3')
axes[1, 1].plot(t_window, I0, 'purple', linewidth=2, label='|I₀|')
axes[1, 1].axvline(x=0, color='magenta', linestyle='--')
axes[1, 1].set_ylabel('Current, kA')
axes[1, 1].set_title('Residual waveform and its fundamental magnitude')
axes[1, 1].legend()
axes[1, 1].grid(True)

# Exclude zero padding from the post-fault RMS summaries.
post_fault_idx = N_PRE
post_fault_end = N_PRE + min(N_POST, len(rec.time) - window_center)
I1_rms = np.sqrt(np.mean(I1[post_fault_idx:post_fault_end]**2) / 2)
I2_rms = np.sqrt(np.mean(I2[post_fault_idx:post_fault_end]**2) / 2)
I0_rms = np.sqrt(np.mean(I0[post_fault_idx:post_fault_end]**2) / 2)

ratio_21 = I2_rms / (I1_rms + 1e-10)
ratio_01 = I0_rms / (I1_rms + 1e-10)

axes[2, 0].bar(['I₂/I₁', 'I₀/I₁'], [ratio_21, ratio_01], color=['orange', 'purple'])
axes[2, 0].axhline(y=1.0, color='red', linestyle='--')
axes[2, 0].set_ylabel('Ratio')
axes[2, 0].set_title('Ratios (fault type features)')
axes[2, 0].grid(True, axis='y')

axes[2, 1].axis('off')

plt.tight_layout()
plt.show()

print(f"\nI₁ (RMS): {I1_rms:.3f} kA")
print(f"I₂ (RMS): {I2_rms:.3f} kA")
print(f"I₀ (RMS): {I0_rms:.3f} kA")
print(f"I₂/I₁ = {ratio_21:.3f}")
print(f"I₀/I₁ = {ratio_01:.3f}")


## 9. Inspect a heuristic fault-type estimate
This optional diagnostic does not add classification channels to the tensor.


In [ ]:
def classify_fault_type(I1_rms, I2_rms, I0_rms):
    """Fault type classification"""
    ratio_21 = I2_rms / (I1_rms + 1e-10)
    ratio_01 = I0_rms / (I1_rms + 1e-10)

    if ratio_21 < 0.15 and ratio_01 < 0.15:
        fault_type = '3ph'
        one_hot = np.array([1, 0, 0, 0])
    elif ratio_21 > 0.7 and ratio_01 < 0.25:
        fault_type = '2ph'
        one_hot = np.array([0, 1, 0, 0])
    elif ratio_21 > 0.7 and ratio_01 > 0.7:
        fault_type = '1ph-G'
        one_hot = np.array([0, 0, 0, 1])
    elif 0.25 < ratio_21 < 0.7 and 0.25 < ratio_01 < 0.7:
        fault_type = '2ph-G'
        one_hot = np.array([0, 0, 1, 0])
    else:
        fault_type = 'undefined'
        one_hot = np.array([0.25, 0.25, 0.25, 0.25])

    return fault_type, one_hot, (ratio_21, ratio_01)

fault_type, one_hot, ratios = classify_fault_type(I1_rms, I2_rms, I0_rms)
ratio_21, ratio_01 = ratios

print(f"✓ Classification: {fault_type}")


In [ ]:
# Visualize the classification
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].set_xlim([0, 1.5])
axes[0].set_ylim([0, 1.5])

from matplotlib.patches import Rectangle
axes[0].add_patch(Rectangle((0, 0), 0.15, 0.15, facecolor='green', alpha=0.3))
axes[0].add_patch(Rectangle((0.7, 0), 0.8, 0.25, facecolor='blue', alpha=0.3))
axes[0].add_patch(Rectangle((0.7, 0.7), 0.8, 0.8, facecolor='red', alpha=0.3))
axes[0].add_patch(Rectangle((0.25, 0.25), 0.45, 0.45, facecolor='orange', alpha=0.3))

axes[0].scatter([ratio_21], [ratio_01], s=300, c='magenta', marker='*', edgecolors='black', linewidths=2)
axes[0].set_xlabel('I₂/I₁')
axes[0].set_ylabel('I₀/I₁')
axes[0].set_title('Decision regions')
axes[0].grid(True)

labels = ['3ph', '2ph', '2ph-G', '1ph-G']
colors = ['green', 'blue', 'orange', 'red']
bars = axes[1].bar(labels, one_hot, color=colors, edgecolor='black', linewidth=2)

for bar, val in zip(bars, one_hot):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height + 0.02, f'{val:.2f}',
                ha='center', va='bottom', fontsize=12, fontweight='bold')

axes[1].set_ylabel('Class indicator (heuristic)')
axes[1].set_title(f'Result: {fault_type}')
axes[1].set_ylim([0, 1.2])
axes[1].grid(True, axis='y')

plt.tight_layout()
plt.show()

print(f"One-hot: {one_hot}")


## 10. Normalize by physical base values


In [ ]:
# Per-unit normalization using the line parameters from setup.
if S_BASE_MVA <= 0 or LINE_PARAMS['Unom_kv'] <= 0 or LINE_PARAMS['L_km'] <= 0:
    raise ValueError("Base power, nominal voltage, and line length must be positive.")
if FAULT_DISTANCE_KM is not None and not 0 <= FAULT_DISTANCE_KM <= LINE_PARAMS['L_km']:
    raise ValueError("FAULT_DISTANCE_KM must be between zero and the line length.")

U_base_kV = LINE_PARAMS['Unom_kv'] / np.sqrt(3)
I_base_kA = S_BASE_MVA / (np.sqrt(3) * LINE_PARAMS['Unom_kv'])

print("=== BASE QUANTITIES ===")
print(f"U_base = {U_base_kV:.2f} kV (phase voltage)")
print(f"I_base = {I_base_kA:.3f} kA")
print(f"Z₁ = {np.sqrt((LINE_PARAMS['r1_ohm_km']*LINE_PARAMS['L_km'])**2 + (LINE_PARAMS['x1_ohm_km']*LINE_PARAMS['L_km'])**2):.2f} ohm")

IA_pu = IA_window / I_base_kA
IB_pu = IB_window / I_base_kA
IC_pu = IC_window / I_base_kA
UA_pu = UA_window / U_base_kV
UB_pu = UB_window / U_base_kV
UC_pu = UC_window / U_base_kV

I0_pu = I0 / I_base_kA
I1_pu = I1 / I_base_kA
I2_pu = I2 / I_base_kA
U0_pu = U0 / U_base_kV
U1_pu = U1 / U_base_kV
U2_pu = U2 / U_base_kV

if FAULT_DISTANCE_KM is not None and LINE_PARAMS['L_km'] > 0:
    d_target_pu = FAULT_DISTANCE_KM / LINE_PARAMS['L_km']
    print(f"\nTarget: {FAULT_DISTANCE_KM:.1f} km = {d_target_pu:.3f} p.u.")
else:
    d_target_pu = None
    print("\nTarget: not set (FAULT_DISTANCE_KM = None)")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(t_window, I1_pu, 'b-', label='I₁')
axes[0, 0].plot(t_window, I2_pu, 'orange', label='I₂')
axes[0, 0].plot(t_window, I0_pu, 'purple', label='I₀')
axes[0, 0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
axes[0, 0].axvline(x=0, color='magenta', linestyle='--')
axes[0, 0].set_ylabel('Current, p.u.')
axes[0, 0].set_title('Sequence current magnitudes in p.u.')
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(t_window, U1_pu, 'b-', label='U₁')
axes[0, 1].plot(t_window, U2_pu, 'orange', label='U₂')
axes[0, 1].plot(t_window, U0_pu, 'purple', label='U₀')
axes[0, 1].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
axes[0, 1].axvline(x=0, color='magenta', linestyle='--')
axes[0, 1].set_ylabel('Voltage, p.u.')
axes[0, 1].set_title('Sequence voltage magnitudes in p.u.')
axes[0, 1].legend()
axes[0, 1].grid(True)

categories = ['I₁', 'I₂', 'I₀', 'U₁', 'U₂', 'U₀']
after = [np.max(np.abs(I1_pu)), np.max(np.abs(I2_pu)), np.max(np.abs(I0_pu)),
         np.max(np.abs(U1_pu)), np.max(np.abs(U2_pu)), np.max(np.abs(U0_pu))]

x = np.arange(len(categories))
axes[1, 0].bar(x, after, color=['blue', 'orange', 'purple', 'blue', 'orange', 'purple'])
axes[1, 0].set_ylabel('Amplitude, p.u.')
axes[1, 0].set_title('Maximum values in p.u.')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(categories)
axes[1, 0].grid(True, axis='y')

if d_target_pu is not None:
    axes[1, 1].barh(['Distance'], [d_target_pu], color='green', edgecolor='black', height=0.3)
    axes[1, 1].set_xlim([0, 1])
    axes[1, 1].set_xlabel('p.u.')
    axes[1, 1].set_title(f'Target: {d_target_pu:.3f} p.u.')
    axes[1, 1].text(d_target_pu/2, 0, f'{FAULT_DISTANCE_KM:.1f} km',
                    ha='center', va='center', fontsize=14, fontweight='bold', color='white')
    axes[1, 1].set_yticks([])
else:
    axes[1, 1].text(0.5, 0.5, 'Target not set', ha='center', va='center', transform=axes[1, 1].transAxes, fontsize=14)
    axes[1, 1].set_title('Target: not set')

plt.tight_layout()
plt.show()

print(f"Max I₁ after the window center: {np.max(np.abs(I1_pu[N_PRE:post_fault_end])):.2f} p.u.")
print(f"Min U₁ after the window center: {np.min(np.abs(U1_pu[N_PRE:post_fault_end])):.2f} p.u.")


## 11. Build and inspect the final tensor
Channel order: IA, IB, IC, UA, UB, UC, |I1|, |I2|, |I0|, |U1|, |U2|, |U0|.


In [ ]:
# Notebook tensor layout: all phase channels, then sequence magnitudes.
X = np.array([
    IA_pu, IB_pu, IC_pu, UA_pu, UB_pu, UC_pu,
    I1_pu, I2_pu, I0_pu, U1_pu, U2_pu, U0_pu,
], dtype=np.float32)
if X.shape != (12, L_WINDOW) or not np.isfinite(X).all():
    raise ValueError(f"Invalid output tensor: shape={X.shape}, expected (12, {L_WINDOW})")

y = d_target_pu if d_target_pu is not None else np.nan

print("✓ Final tensor created")
print(f"  X shape: {X.shape} (channels × time)")
print("  Channels: IA, IB, IC, UA, UB, UC, |I₁|, |I₂|, |I₀|, |U₁|, |U₂|, |U₀|")
if d_target_pu is not None:
    print(f"  Target variable y: {y:.4f} p.u.")
else:
    print("  Target variable y: not set (set FAULT_DISTANCE_KM)")


In [ ]:
# Visualize the final tensor
fig, axes = plt.subplots(6, 2, figsize=(14, 18))
axes = axes.flatten()

channel_names = [
    'IA (phase A current)', 'IB (phase B current)', 'IC (phase C current)',
    'UA (phase A voltage)', 'UB (phase B voltage)', 'UC (phase C voltage)',
    '|I₁| (positive-sequence current)', '|I₂| (negative-sequence current)', '|I₀| (zero-sequence current)',
    '|U₁| (positive-sequence voltage)', '|U₂| (negative-sequence voltage)', '|U₀| (zero-sequence voltage)'
]

colors = ['#d62728', '#2ca02c', '#1f77b4',
          '#d62728', '#2ca02c', '#1f77b4',
          'blue', 'orange', 'purple',
          'blue', 'orange', 'purple']

for i in range(12):
    axes[i].plot(t_window, X[i], color=colors[i], linewidth=1.5)
    axes[i].axvline(x=0, color='magenta', linestyle='--', alpha=0.7)
    axes[i].set_ylabel(f'Channel {i}')
    axes[i].set_title(channel_names[i])
    axes[i].grid(True)

target_label = f'{y:.3f} p.u.' if d_target_pu is not None else 'not provided'
fig.suptitle(f'Final tensor X ∈ R^(12×{L_WINDOW})\nTarget: {target_label}',
             fontsize=14, fontweight='bold')

for i in [10, 11]:
    axes[i].set_xlabel('Time from window center, ms')

plt.tight_layout(rect=[0, 0, 1, 0.98])
fig.savefig(OUTPUT_FOLDER / 'final_tensor.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n{'Channel':<8} {'Name':<25} {'Range':<20}")
print("-" * 55)
for i, name in enumerate(channel_names):
    min_val, max_val = np.min(X[i]), np.max(X[i])
    print(f"{i:<8} {name:<25} [{min_val:6.2f}, {max_val:6.2f}]")

if d_target_pu is not None:
    print(f"\nTarget variable: y = {y:.4f} p.u. = {y*100:.1f}% of the line length")
else:
    print("\nTarget not provided; y is stored as NaN.")


## 12. Save the tensor and metadata

The NPZ contains `X`, `y`, and a JSON string `metadata`; load it with
`np.load(path, allow_pickle=False)` and decode metadata with
`json.loads(str(saved["metadata"]))`. `final_tensor.png` is saved at 300 DPI.


In [ ]:
output_data = {
    'X': X,
    'y': y,
    'metadata': {
        'source_record': RECORD_NAME,
        'fault_type_heuristic': fault_type,
        'window_source': window_source,
        'window_center_sample': window_center,
        'detected_fault_sample': int(fault_idx_detected) if fault_idx_detected is not None else None,
        'phase_units': 'p.u.',
        'sequence_representation': 'one-cycle DFT peak magnitudes in p.u.',
        'fault_distance_km': FAULT_DISTANCE_KM,
        'line_length_km': LINE_PARAMS['L_km'],
        'sample_rate': data['fs'],
        'grid_frequency_hz': F_NET,
        'base_power_mva': S_BASE_MVA,
        'nominal_line_voltage_kv': LINE_PARAMS['Unom_kv'],
        'pre_window_samples': N_PRE,
        'post_window_samples': N_POST,
        'padding_left_samples': max(0, N_PRE - window_center),
        'padding_right_samples': max(0, window_center + N_POST - len(rec.time)),
        'current_channel_numbers': i_p_idx,
        'voltage_channel_numbers': u_p_idx,
        'window_samples': L_WINDOW,
        'channel_names': channel_names,
        'tensor_description': 'X ∈ R^(12 × L_window): [IA, IB, IC, UA, UB, UC, |I1|, |I2|, |I0|, |U1|, |U2|, |U0|]'
    }
}

output_data['metadata']['source_provenance'] = source_metadata

output_path = OUTPUT_FOLDER / 'fault_sample_processed.npz'
np.savez_compressed(
    output_path, X=X, y=y, metadata=json.dumps(output_data['metadata'])
)
with np.load(output_path, allow_pickle=False) as saved:
    np.testing.assert_array_equal(saved['X'], X)
    assert saved['X'].shape == (12, L_WINDOW)
print(f"Pipeline complete. Tensor: {X.shape}, dtype: {X.dtype}")
print(f"Saved tensor: {output_path}")
print(f"Saved plot:   {OUTPUT_FOLDER / 'final_tensor.png'}")
print(f"Filtered COMTRADE: {file_name}.cfg / {file_name}.dat")


## 13. Estimate distance from the in-memory test waveforms

Run these cells after preprocessing; no CSV reload or trained model is required.
They use `IA_window`…`UC_window` in **kA/kV**, retain phase angles, and exclude
zero padding. `X` remains available as the separate normalized feature tensor.

The single-ended **simple reactance** estimate is
$d_{km}=\operatorname{Im}(U_{loop}/I_{loop})/x_1$, with $x_1$ in Ω/km.
Phase-pair loops use voltage/current differences. Ground loops use
$I_{loop}=I_{phase}+k_0(I_A+I_B+I_C)$, where
$k_0=(Z_0-Z_1)/(3Z_1)$ when both zero-sequence parameters are supplied.
`POS` uses positive-sequence voltage/current for a balanced three-phase fault.

Set `FAULT_LOOP` explicitly when the heuristic fault classification is uncertain.
For ground faults without `r0_ohm_km` and `x0_ohm_km`, the result is labeled an
**uncompensated apparent distance**. Replace the example line parameters with the
actual line values before interpreting it as a physical location. Fault resistance,
load, remote infeed, and measurement/line-parameter errors can bias this method;
cycle-to-cycle variation is not an accuracy bound.

Reference: [SEL, Impedance-Based Fault Location Experience, Table 1 and Simple
Reactance Method](https://selinc.com/api/download/4912/?lang=en).


In [ ]:
def fit_fundamental_phasors(samples: np.ndarray, fs: float, f0: float) -> np.ndarray:
    """Fit fundamental RMS phasors to (channels, time) signals with a DC offset.

    A waveform sqrt(2)*abs(P)*cos(2*pi*f0*t + angle(P)) returns phasor P.
    All channels must share the same sample times and reference direction.
    """
    values = np.asarray(samples, dtype=float)
    if values.ndim != 2 or values.shape[1] < 3 or not np.isfinite(values).all():
        raise ValueError("Phasor inputs must be finite (channels, time) arrays.")
    if not np.isfinite([fs, f0]).all() or not 0 < f0 < fs / 2:
        raise ValueError("Require finite sampling/grid frequencies with 0 < f0 < fs/2.")
    angle = 2 * np.pi * f0 * np.arange(values.shape[1]) / fs
    basis = np.column_stack([np.cos(angle), np.sin(angle), np.ones_like(angle)])
    coefficients, _, rank, _ = np.linalg.lstsq(basis, values.T, rcond=None)
    if rank != 3:
        raise ValueError("The selected samples cannot resolve a fundamental phasor.")
    return (coefficients[0] - 1j * coefficients[1]) / np.sqrt(2)


def estimate_impedance_distance(
    currents_ka: np.ndarray,
    voltages_kv: np.ndarray,
    fs: float,
    f0: float,
    center_idx: int,
    measured_stop: int,
    line_params: dict,
    fault_loop: str,
    settle_cycles: float = 1.0,
    average_cycles: int = 3,
    min_current_ka: float = 1e-3,
) -> dict:
    """Estimate apparent distance using real post-fault cycles already in memory.

    Uses d_km = imag(U_loop / I_loop) / x1_ohm_km. For ground loops,
    I_loop = I_phase + k0*(IA+IB+IC), k0=(Z0-Z1)/(3*Z1), if Z0 is supplied.
    Missing Z0 yields an explicitly uncompensated apparent distance.
    The caller supplies the exclusive end of measured data to exclude padding.
    """
    currents = np.asarray(currents_ka, dtype=float)
    voltages = np.asarray(voltages_kv, dtype=float)
    if currents.shape != voltages.shape or currents.ndim != 2 or currents.shape[0] != 3:
        raise ValueError("Current and voltage arrays must both have shape (3, N).")
    if not np.isfinite(currents).all() or not np.isfinite(voltages).all():
        raise ValueError("Waveforms must contain only finite values.")
    if not np.isfinite([fs, f0]).all() or not 0 < f0 < fs / 2:
        raise ValueError("Require finite sampling/grid frequencies with 0 < f0 < fs/2.")
    if not isinstance(center_idx, (int, np.integer)) or not isinstance(
        measured_stop, (int, np.integer)
    ):
        raise ValueError("Window bounds must be integer sample indices.")
    if not 0 <= center_idx < measured_stop <= currents.shape[1]:
        raise ValueError("Measured post-fault bounds must lie inside the phase window.")
    if not np.isfinite(settle_cycles) or settle_cycles < 0:
        raise ValueError("settle_cycles must be finite and nonnegative.")
    if isinstance(average_cycles, bool) or not isinstance(average_cycles, (int, np.integer)):
        raise ValueError("average_cycles must be a positive integer.")
    if average_cycles < 1 or not np.isfinite(min_current_ka) or min_current_ka <= 0:
        raise ValueError("Cycle count and minimum loop current must be positive.")

    r1 = float(line_params['r1_ohm_km'])
    x1 = float(line_params['x1_ohm_km'])
    line_length = float(line_params['L_km'])
    if not np.isfinite([r1, x1, line_length]).all() or r1 < 0 or x1 <= 0 or line_length <= 0:
        raise ValueError("Require finite r1 >= 0, x1 > 0, and line length > 0.")
    z1 = complex(r1, x1)
    loop = fault_loop.upper()
    if loop not in {'AG', 'BG', 'CG', 'AB', 'BC', 'CA', 'POS'}:
        raise ValueError("Choose AG, BG, CG, AB, BC, CA, or POS.")

    ground_loop = loop.endswith('G')
    r0, x0 = line_params.get('r0_ohm_km'), line_params.get('x0_ohm_km')
    if ground_loop and (r0 is None) != (x0 is None):
        raise ValueError("Ground compensation requires both r0_ohm_km and x0_ohm_km.")
    compensated = ground_loop and r0 is not None and x0 is not None
    k0 = 0j
    if compensated:
        r0, x0 = float(r0), float(x0)
        if not np.isfinite([r0, x0]).all() or r0 < 0 or x0 <= 0:
            raise ValueError("Require finite zero-sequence parameters r0 >= 0 and x0 > 0.")
        k0 = (complex(r0, x0) - z1) / (3 * z1)

    cycle_samples = int(np.ceil(fs / f0))
    first = center_idx + int(np.ceil(settle_cycles * fs / f0))
    cycle_count = min(average_cycles, max(0, (measured_stop - first) // cycle_samples))
    if cycle_count == 0:
        raise ValueError("Less than one measured post-fault cycle remains after settling.")

    rows = []
    skipped = 0
    phase_indices = {'A': 0, 'B': 1, 'C': 2}
    a = np.exp(2j * np.pi / 3)
    positive_sequence = np.array([1, a, a**2]) / 3
    for cycle in range(cycle_count):
        start = first + cycle * cycle_samples
        stop = start + cycle_samples
        phasors = fit_fundamental_phasors(
            np.vstack([currents[:, start:stop], voltages[:, start:stop]]), fs, f0
        )
        i_phase, u_phase = phasors[:3], phasors[3:]
        if ground_loop:
            phase = phase_indices[loop[0]]
            i_loop = i_phase[phase] + k0 * i_phase.sum()
            u_loop = u_phase[phase]
        elif loop == 'POS':
            i_loop = positive_sequence @ i_phase
            u_loop = positive_sequence @ u_phase
        else:
            left, right = phase_indices[loop[0]], phase_indices[loop[1]]
            i_loop = i_phase[left] - i_phase[right]
            u_loop = u_phase[left] - u_phase[right]
        if abs(i_loop) < min_current_ka:
            skipped += 1
            continue
        z_apparent = u_loop / i_loop  # kV / kA = ohm; no base-value conversion.
        distance_km = float(z_apparent.imag / x1)
        if not np.isfinite(distance_km):
            raise ValueError("The apparent impedance did not produce a finite distance.")
        rows.append({
            'cycle': cycle + 1,
            'start_sample': int(start),
            'stop_sample': int(stop),
            'start_after_fault_ms': float((start - center_idx) / fs * 1000),
            'stop_after_fault_ms': float((stop - center_idx) / fs * 1000),
            'loop_current_rms_ka': float(abs(i_loop)),
            'loop_voltage_rms_kv': float(abs(u_loop)),
            'r_apparent_ohm': float(z_apparent.real),
            'x_apparent_ohm': float(z_apparent.imag),
            'distance_km': distance_km,
        })
    if not rows:
        raise ValueError("Every selected cycle has insufficient loop current for division.")

    distances = np.array([row['distance_km'] for row in rows])
    estimate = float(np.median(distances))
    return {
        'method': 'single-ended simple reactance',
        'fault_loop': loop,
        'estimated_distance_km': estimate,
        'distance_percent_of_line': float(100 * estimate / line_length),
        'within_line': bool(0 <= estimate <= line_length),
        'ground_compensated': bool(compensated),
        'uncompensated_ground_estimate': bool(ground_loop and not compensated),
        'k0_real': float(k0.real),
        'k0_imag': float(k0.imag),
        'cycle_min_km': float(distances.min()),
        'cycle_max_km': float(distances.max()),
        'cycles_used': len(rows),
        'cycles_skipped_low_current': skipped,
        'cycles': rows,
    }


In [ ]:
# Run this cell after the preprocessing cells. All waveforms are read from memory.
FAULT_LOOP = 'AUTO'  # AUTO, AG, BG, CG, AB, BC, CA, or POS (balanced three-phase).
DISTANCE_SETTLE_CYCLES = 1.0
DISTANCE_AVERAGE_CYCLES = 3
MIN_LOOP_CURRENT_KA = 1e-3
# For a compensated ground-fault estimate, set both values in LINE_PARAMS:
# LINE_PARAMS['r0_ohm_km'] = ...
# LINE_PARAMS['x0_ohm_km'] = ...

estimated_distance_km = None
impedance_result = None
distance_cycle_table = pd.DataFrame()

try:
    if FAULT_TIME_S is None and fault_idx_detected is None:
        raise ValueError('No fault inception was identified; set FAULT_TIME_S before estimation.')
    current_window_ka = np.stack([IA_window, IB_window, IC_window])
    voltage_window_kv = np.stack([UA_window, UB_window, UC_window])
    measured_stop = min(L_WINDOW, N_PRE + len(rec.time) - window_center)
    fault_loop_used = FAULT_LOOP.upper()
    loop_selection = 'manual'
    if fault_loop_used == 'AUTO':
        measured_currents = current_window_ka[:, N_PRE:measured_stop]
        phase_rms = np.sqrt(np.mean(measured_currents**2, axis=1))
        if fault_type == '1ph-G':
            fault_loop_used = 'ABC'[int(np.argmax(phase_rms))] + 'G'
        elif fault_type in {'2ph', '2ph-G'}:
            phases = frozenset(np.argsort(phase_rms)[-2:].tolist())
            fault_loop_used = {
                frozenset([0, 1]): 'AB',
                frozenset([1, 2]): 'BC',
                frozenset([0, 2]): 'CA',
            }[phases]
        elif fault_type == '3ph':
            fault_loop_used = 'POS'
        else:
            raise ValueError('Fault type is uncertain; select FAULT_LOOP explicitly.')
        loop_selection = f'heuristic AUTO: {fault_type}, phase RMS {phase_rms.round(4)} kA'

    impedance_result = estimate_impedance_distance(
        current_window_ka, voltage_window_kv, FS, F_NET, N_PRE, measured_stop,
        LINE_PARAMS, fault_loop_used,
        settle_cycles=DISTANCE_SETTLE_CYCLES,
        average_cycles=DISTANCE_AVERAGE_CYCLES,
        min_current_ka=MIN_LOOP_CURRENT_KA,
    )
    estimated_distance_km = impedance_result['estimated_distance_km']
    impedance_result.update({
        'source_record': RECORD_NAME,
        'loop_selection': loop_selection,
        'inception_source': window_source,
        'line_parameters': dict(LINE_PARAMS),
        'sampling_frequency_hz': FS,
        'grid_frequency_hz': F_NET,
        'true_distance_km': FAULT_DISTANCE_KM,
    })
    distance_cycle_table = pd.DataFrame(impedance_result['cycles'])
    estimate_label = 'Estimated distance'
    if impedance_result['uncompensated_ground_estimate']:
        estimate_label = 'Uncompensated apparent distance'
    print(f'{estimate_label}: {estimated_distance_km:.4f} km from the measurement terminal')
    print(f'Loop: {fault_loop_used} ({loop_selection})')
    print(f"Line fraction: {impedance_result['distance_percent_of_line']:.2f}%")
    print(f"Cycles used: {impedance_result['cycles_used']}; "
          f"range {impedance_result['cycle_min_km']:.4f}–"
          f"{impedance_result['cycle_max_km']:.4f} km (cycle variation, not uncertainty)")
    if impedance_result['uncompensated_ground_estimate']:
        print('r0/x0 are not set: ground-current compensation was omitted.')
    if not impedance_result['within_line']:
        print('The estimate lies outside the configured line; it has not been clipped.')
    if FAULT_DISTANCE_KM is not None:
        error_km = abs(estimated_distance_km - FAULT_DISTANCE_KM)
        impedance_result['absolute_error_km'] = float(error_km)
        print(f'True distance: {FAULT_DISTANCE_KM:.4f} km; absolute error: {error_km:.4f} km')
    print('Uses the configured line impedances; the setup values are examples until confirmed.')
    display(distance_cycle_table)
except ValueError as error:
    impedance_result = {'status': 'unavailable', 'reason': str(error), 'source_record': RECORD_NAME}
    print(f'Fault-distance estimate unavailable: {error}')

# Save the result separately; the optional ground-truth target y remains unchanged.
(OUTPUT_FOLDER / 'fault_distance_estimate.json').write_text(
    json.dumps(impedance_result, indent=2, allow_nan=False) + '\n'
)
distance_cycle_table.to_csv(OUTPUT_FOLDER / 'fault_distance_cycles.csv', index=False)
print(f"Result saved: {OUTPUT_FOLDER / 'fault_distance_estimate.json'}")


## 14. Visualize the fault-distance calculation

Run this cell after the estimator. It uses the current `impedance_result` to draw an
illustrative radial single-line diagram, the measurement terminal, estimated fault
location, actual distance scale, and calculation steps. The schematic geometry is
not to scale. Ground compensation and unavailable/out-of-line results remain explicit.

The figure is displayed here and saved as `fault_distance_diagram.png` (300 DPI)
and `fault_distance_diagram.svg` in the current output folder.


In [ ]:
from pathlib import Path
from textwrap import fill

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Circle, FancyBboxPatch, Polygon


def plot_fault_distance_diagram(result: dict | None) -> plt.Figure:
    """Draw an illustrative radial feeder and the calculation from a result snapshot."""
    result = result or {}
    params = result.get('line_parameters', {})
    distance = result.get('estimated_distance_km')
    length = params.get('L_km')
    valid = (
        result.get('status') != 'unavailable'
        and distance is not None and length is not None
        and np.isfinite([distance, length]).all() and length > 0
    )
    inside = bool(valid and 0 <= distance <= length)
    loop = result.get('fault_loop', '?')
    uncompensated = result.get('uncompensated_ground_estimate', False)
    ink, muted, teal, fault = '#23364A', '#64748B', '#147D92', '#C84930'
    pale, edge = '#F4F8FB', '#DCE5ED'
    fig = plt.figure(figsize=(16, 9.5), facecolor='white')
    fig.text(0.045, 0.955, 'Fault distance from terminal measurements',
             fontsize=23, weight='bold', color=ink)
    record = result.get('source_record', 'no estimate loaded')
    fig.text(0.045, 0.918,
             f'Record: {record}  |  Single-ended reactance method  |  Loop: {loop}',
             fontsize=11, color=muted)

    ax = fig.add_axes([0.035, 0.45, 0.93, 0.43])
    ax.set(xlim=(0, 16), ylim=(0, 6))
    ax.set_aspect('equal')
    ax.axis('off')
    y_line, terminal, remote = 3.5, 3.1, 13.0
    fault_x = 7.6 if inside and 0 < distance < length else terminal
    if inside and distance == length:
        fault_x = remote

    # Illustrative topology only: no network model is inferred from the oscillogram.
    ax.add_patch(Circle((1.0, y_line), 0.48, fill=False, lw=2, ec=ink))
    wave_x = np.linspace(0.65, 1.35, 50)
    ax.plot(wave_x, y_line + 0.17 * np.sin((wave_x - 0.65) / 0.70 * 2 * np.pi),
            color=ink, lw=1.5)
    ax.plot([1.48, terminal], [y_line, y_line], color=ink, lw=2.5)
    ax.text(1, 4.45, 'Source /\nsubstation', ha='center', color=ink, fontsize=11)
    ax.plot([terminal, terminal], [3.05, 4.1], color=ink, lw=4)
    ax.text(terminal, 4.45, 'Measurement terminal\n0 km', ha='center',
            color=ink, fontsize=11, weight='bold')
    ax.plot([terminal, remote], [y_line, y_line], color=muted, lw=3)
    if inside:
        ax.plot([terminal, fault_x], [y_line, y_line], color=fault, lw=4)
    ax.plot([remote, remote], [3.05, 4.1], color=ink, lw=4)
    ax.plot([remote, 14.45], [y_line, y_line], color=ink, lw=2.5)
    ax.add_patch(Polygon([[14.45, 3.83], [15.05, 3.5], [14.45, 3.17]],
                         closed=True, fc='white', ec=ink, lw=2))
    end_label = f'Remote bus\n{length:g} km' if valid else 'Remote bus'
    ax.text(remote, 4.45, end_label, ha='center', color=ink, fontsize=11)
    ax.text(14.7, 2.8, 'Downstream load', ha='center', color=muted, fontsize=10)

    # CT on the feeder and VT in shunt at the same measurement terminal.
    ax.add_patch(Circle((3.55, y_line), 0.20, fill=False, ec=teal, lw=2))
    ax.text(3.65, 3.93, 'CT', color=teal, fontsize=10)
    ax.plot([terminal, terminal], [3.05, 2.83], color=teal, lw=1.5)
    for y_circle in (2.65, 2.39):
        ax.add_patch(Circle((terminal, y_circle), 0.18, fill=False, ec=teal, lw=1.5))
    ax.plot([terminal, terminal], [2.21, 2.07], color=teal, lw=1.3)
    for offset, width in ((0, 0.22), (0.08, 0.15), (0.16, 0.07)):
        ax.plot([terminal - width, terminal + width], [2.07 - offset] * 2,
                color=teal, lw=1.3)
    ax.text(3.42, 2.43, 'VT', color=teal, fontsize=10)
    ax.plot([3.55, 4.2, 4.2], [3.25, 2.7, 1.8], '--', color=teal, lw=1.2)
    ax.plot([3.28, 4.0, 4.0], [2.52, 2.2, 1.8], '--', color=teal, lw=1.2)
    ax.add_patch(FancyBboxPatch((2.15, 0.88), 3.7, 0.8,
                               boxstyle='round,pad=0.14', fc=pale, ec=edge))
    ax.text(4.0, 1.29, 'Recorded $U_{ABC}$, $I_{ABC}$\nRMS phasors from measured cycles',
            ha='center', va='center', color=ink, fontsize=10)

    if inside:
        ax.scatter([fault_x], [y_line], s=75, color=fault, zorder=5)
        if 0 < distance < length:
            ax.plot([fault_x, fault_x - 0.20, fault_x + 0.16, fault_x - 0.05],
                    [3.45, 2.95, 2.95, 2.45], color=fault, lw=2.5)
            if loop in {'AG', 'BG', 'CG'}:
                for offset, width in ((0, 0.28), (0.10, 0.19), (0.20, 0.09)):
                    ax.plot([fault_x - width, fault_x + width], [2.35 - offset] * 2,
                            color=fault, lw=1.6)
            ax.text(fault_x + 0.40, 2.72, f'Estimated fault\nLoop {loop}',
                    va='center', color=fault, fontsize=11, weight='bold')
        else:
            terminal_name = 'measurement terminal' if distance == 0 else 'remote bus'
            ax.annotate(f'Estimated fault at {terminal_name}\nLoop {loop}',
                        xy=(fault_x, y_line), xytext=(8.0, 2.5), ha='center',
                        color=fault, fontsize=11, weight='bold',
                        arrowprops={'arrowstyle': '->', 'color': fault, 'lw': 1.3})
        if distance > 0:
            ax.annotate('', (fault_x, 5.35), (terminal, 5.35),
                        arrowprops={'arrowstyle': '<->', 'color': fault, 'lw': 1.5})
            ax.text((terminal + fault_x) / 2, 5.56, f'd = {distance:.4f} km',
                    ha='center', color=fault, fontsize=12, weight='bold')
        if distance < length:
            ax.annotate('', (remote, 5.35), (fault_x, 5.35),
                        arrowprops={'arrowstyle': '<->', 'color': muted, 'lw': 1.2})
            ax.text((remote + fault_x) / 2, 5.56, f'L − d = {length-distance:.4f} km',
                    ha='center', color=muted, fontsize=11)
    else:
        message = (f'Estimate: {distance:.4f} km — outside the configured line'
                   if valid else 'No fault location available; run the estimation cell.')
        ax.text(8.8, 5.4, message, ha='center', color=fault, fontsize=12, weight='bold')
    line_label = (f'Configured line: L = {length:g} km' if valid else 'Configured feeder')
    ax.text(10.4, 1.30, line_label + '\nIllustrative topology; schematic not to scale',
            ha='center', va='center', fontsize=10, color=muted)

    # A separate true distance scale prevents schematic spacing from implying a wrong ratio.
    scale = fig.add_axes([0.215, 0.43, 0.69, 0.032])
    scale.set_ylim(-1, 1)
    scale.set_yticks([])
    for spine in ('left', 'top', 'right'):
        scale.spines[spine].set_visible(False)
    scale.spines['bottom'].set_color(edge)
    scale.tick_params(axis='x', labelsize=9, colors=muted)
    if valid:
        margin = max(length, abs(distance), 1) * 0.045
        scale.set_xlim(min(0, distance) - margin, max(length, distance) + margin)
        scale.plot([0, length], [0, 0], color=muted, lw=3)
        scale.scatter([distance], [0], color=fault, marker='v', s=65, zorder=4)
        scale.set_xticks(np.linspace(0, length, 6))
        scale.annotate(f'{distance:.4f} km', (distance, 0), xytext=(0, 12),
                       textcoords='offset points', ha='center', color=fault, fontsize=9)
        scale.set_xlabel('Actual distance scale from the measurement terminal (km)',
                         fontsize=9, color=muted)
    else:
        scale.axis('off')

    # Three panels connect measurements, complex loop impedance, and median distance.
    card_axes = []
    for index, heading in enumerate(('1  Measurements → phasors',
                                     '2  Select the electrical loop',
                                     '3  Reactance → distance')):
        card = fig.add_axes([0.045 + 0.313 * index, 0.115, 0.287, 0.22])
        card.set(xlim=(0, 1), ylim=(0, 1))
        card.axis('off')
        card.add_patch(FancyBboxPatch((0.01, 0.01), 0.98, 0.98,
                                     boxstyle='round,pad=0', fc=pale, ec=edge))
        card.text(0.055, 0.87, heading, color=teal, fontsize=12, weight='bold')
        card_axes.append(card)

    cycles = result.get('cycles', [])
    if valid and cycles:
        cycle = min(cycles, key=lambda row: abs(row['distance_km'] - distance))
        details = (
            f"Example cycle {cycle['cycle']} "
            f"({cycle['start_after_fault_ms']:g}–{cycle['stop_after_fault_ms']:g} ms after t₀)\n"
            f"|U loop| = {cycle['loop_voltage_rms_kv']:.3f} kV RMS\n"
            f"|I loop| = {cycle['loop_current_rms_ka']:.3f} kA RMS\n"
            'Complex phasors retain the voltage/current angle.\n'
            f"{result['cycles_used']} measured cycles; zero padding excluded."
        )
        card_axes[0].text(0.055, 0.68, details, va='top', linespacing=1.7,
                          color=ink, fontsize=9.5)
        if loop in {'AG', 'BG', 'CG'}:
            phase = loop[0]
            loop_text = f'U loop = U{phase}\nI loop = I{phase}'
            if result.get('ground_compensated'):
                loop_text += ' + k₀ (IA + IB + IC)'
            else:
                loop_text += '  (ground compensation omitted)'
        elif loop == 'POS':
            loop_text = 'U loop = U₁\nI loop = I₁ (positive sequence)'
        else:
            loop_text = f'U loop = U{loop[0]} − U{loop[1]}\nI loop = I{loop[0]} − I{loop[1]}'
        impedance = (f"Example Z loop = {cycle['r_apparent_ohm']:.4f} "
                     f"{cycle['x_apparent_ohm']:+.4f}j Ω")
        card_axes[1].text(0.055, 0.69, loop_text + '\nZ loop = U loop / I loop\n' + impedance,
                          va='top', color=ink, fontsize=9.5, linespacing=1.8)
        x1 = params['x1_ohm_km']
        median_x = float(np.median([row['x_apparent_ohm'] for row in cycles]))
        card_axes[2].text(0.055, 0.69,
                          '$d = \\mathrm{median}[\\mathrm{Im}(Z_{loop,c}) / x_1]$\n'
                          f'x₁ = {x1:g} Ω/km\n'
                          f'{median_x:.6f} Ω / {x1:g} Ω/km',
                          va='top', color=ink, fontsize=11, linespacing=1.6)
        card_axes[2].text(0.055, 0.20, f'{distance:.4f} km', fontsize=23,
                          color=fault, weight='bold')
        card_axes[2].text(0.055, 0.07, f'{100*distance/length:.2f}% of the configured line',
                          color=muted, fontsize=9.5)
    else:
        reason = result.get('reason', 'Run Section 13 to create impedance_result.')
        card_axes[0].text(0.055, 0.68, fill(reason, 48), va='top', fontsize=10, color=ink)
        card_axes[1].text(0.055, 0.68, 'Complex voltage/current phasors\nZ loop = U loop / I loop',
                          va='top', color=ink, fontsize=11, linespacing=1.8)
        card_axes[2].text(0.055, 0.68, 'd = median[Im(Z loop) / x₁]\nEstimate unavailable',
                          va='top', color=ink, fontsize=11, linespacing=1.8)

    status = 'Uncompensated apparent distance' if uncompensated else 'Impedance-based estimate'
    if not valid:
        status = 'Estimate unavailable'
    elif not inside:
        status += ' — outside the configured line (not clipped)'
    fig.text(0.045, 0.064, status, color=fault, fontsize=11, weight='bold')
    fig.text(0.045, 0.036,
             'Interpret using actual line parameters. This drawing illustrates a radial '
             'equivalent; it does not reconstruct the network topology.',
             fontsize=9, color=muted)
    return fig


fault_distance_figure = plot_fault_distance_diagram(globals().get('impedance_result'))
if 'OUTPUT_FOLDER' in globals():
    diagram_folder = Path(OUTPUT_FOLDER)
    diagram_folder.mkdir(parents=True, exist_ok=True)
    for diagram_format in ('png', 'svg'):
        diagram_path = diagram_folder / f'fault_distance_diagram.{diagram_format}'
        fault_distance_figure.savefig(diagram_path, dpi=300, bbox_inches='tight')
    print(f'Diagram saved: {diagram_folder / "fault_distance_diagram.png"}')
    print(f'Vector copy:   {diagram_folder / "fault_distance_diagram.svg"}')
plt.show()
